# Load a shipped MicroRTS-DRL agent and play one game

End-to-end demo using `data/agents/UECD-SingleMap-Best` against `RandomBiasedAI`
on `basesWorkers16x16A`. CPU-only — no GPU needed.

**Prereqs** (run once in a terminal):
```bash
bash setup/local.sh             # conda env + RAISocketAI wheel
bash microrts_agent/microrts/build_bridge.sh
pip install jupyter             # if not already in your env
```

Then launch this notebook from the **repo root** (`jupyter lab`) so the relative
paths below resolve.

## 1. Sanity check — the package imports and the bridge is in place

In [ ]:
from microrts_agent.paths import PROJECT_ROOT

agent_dir = PROJECT_ROOT / "data" / "agents" / "UECD-SingleMap-Best"
print("Repo root :", PROJECT_ROOT)
print("Agent dir :", agent_dir)
print("agent.pt  :", (agent_dir / "agent.pt").exists())
print("bridge.jar:", (PROJECT_ROOT / "microrts_agent" / "microrts" / "bridge.jar").exists())

## 2. Load the agent and inspect its training config

In [ ]:
from microrts_agent.architectures.factory import load_agent_from_config

agent, config = load_agent_from_config(str(agent_dir), device="cpu")
agent.eval()

interesting = [
    "architecture",
    "obs_channels",
    "action_nvec",
    "extended_obs",
    "filtered_masks",
    "reserved_obs",
    "total_timesteps",
    "reward_weight",
]
for k in interesting:
    if k in config:
        print(f"  {k:18s} = {config[k]}")

n_params = sum(p.numel() for p in agent.parameters())
print(f"\nTotal parameters: {n_params:,}")

## 3. Play one game against `RandomBiasedAI` via the CLI

This is the canonical way to evaluate a shipped agent. `evaluate` plays both
as P0 and P1 by default, so 1 game per position = 2 games total.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-m",
    "microrts_agent",
    "evaluate",
    "--agent",
    str(agent_dir),
    "--opponent",
    "RandomBiasedAI",
    "--maps",
    "maps/open_competition/basesWorkers16x16A.xml",
    "--nb_games",
    "1",
    "--max_steps",
    "2000",
]
result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True, timeout=300)
print(result.stdout[-1500:])
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr[-1500:])

## 4. (Optional) Play against `CoacAI` — much stronger opponent, ~30s per game

Uncomment the line below to run it. Expected: UECD-SingleMap-Best wins ~95%
of games against CoacAI on this map (see `data/tournaments/single_map/`).

In [ ]:
# cmd_coac = cmd[:-7] + [
#     '--opponent', 'CoacAI',
#     '--maps', 'maps/open_competition/basesWorkers16x16A.xml',
#     '--nb_games', '1',
#     '--max_steps', '4000',
# ]
# subprocess.run(cmd_coac, cwd=str(PROJECT_ROOT), timeout=600)

## Next steps

- **Recorded gameplay**: 36 mp4 clips under `data/recordings/` (UECD-Best vs the
  full opponent pool on `basesWorkers16x16A`).
- **Tournament results**: `data/tournaments/single_map/` and `multi_map/` ship
  the headline thesis comparison — CSV + ranking PDFs.
- **Other shipped agents**: list them with `ls data/agents/`. Each carries its
  own `config.json` so the snippet in §2 works for any of them.
- **Train your own**: `python -m microrts_agent train --help` — start small with
  `--total-timesteps 100000 --num-bot-envs 8` for a smoke run.